In [99]:
"""
Interactive Markov State Model builder: Load count matrices and convert to transition matrices.
"""
import numpy as np
from pathlib import Path
from deeptime.markov.msm import MarkovStateModel
from deeptime.markov.msm import MaximumLikelihoodMSM
from deeptime.markov import TransitionCountModel

np.set_printoptions(linewidth=np.inf, precision=6, suppress=True)

def load_count_matrix(beta: float, h: float, rc: str, m: int, c_matrix_in: str | None = None) -> np.ndarray:
    """Load a specific count matrix from NPZ file."""
    if c_matrix_in is not None:
        data_path = Path(c_matrix_in)
    else:
        data_path = Path("data") / f"C_matrices_{beta:.3f}_{h:.3f}_{rc}.npz"
        if not data_path.exists() and rc == "cnn":
            legacy = Path("data") / f"C_matrices_{beta:.3f}_{h:.3f}.npz"
            if legacy.exists():
                data_path = legacy

    if not data_path.exists():
        raise FileNotFoundError(f"Missing count-matrix file: {data_path}")

    data = np.load(str(data_path))
    
    # Get list of available lags
    m_list = sorted(int(k.split("m")[-1]) for k in data.keys() if k.startswith("C_m"))
    
    if m not in m_list:
        print(f"Available lags: {m_list}")
        raise ValueError(f"Lag m={m} not found in count matrix file")
    
    return np.asarray(data[f"C_m{m}"], dtype=float)


def count_to_transition(C: np.ndarray, normalize_rows: bool = True) -> np.ndarray:
    """
    Convert count matrix to transition matrix.
    
    Args:
        C: Count matrix (n_states, n_states)
        normalize_rows: If True, normalize by row sums. If False, normalize by total sum.
    
    Returns:
        Transition matrix P
    """
    C = np.asarray(C, dtype=float)
    
    if normalize_rows:
        # Row normalization: P[i,j] = C[i,j] / sum(C[i,:])
        row_sums = C.sum(axis=1, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            P = np.where(row_sums > 0, C / row_sums, 0.0)
    else:
        # Column normalization: P[i,j] = C[i,j] / sum(C[:,j])
        col_sums = C.sum(axis=0, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            P = np.where(col_sums > 0, C / col_sums, 0.0)
    
    return P


def load_brute_force_rate(beta: float, h: float, nucleation_dir: str = "data") -> float:
    """
    Load brute force nucleation rate from nucleation rate files.
    
    Args:
        beta: Beta parameter
        h: H parameter
        nucleation_dir: Directory containing nucleation rate files
    
    Returns:
        Nucleation rate (float)
    """
    # Try to load from nucleation_rates NPZ file
    nuc_path = Path(nucleation_dir) / f"nucleation_{beta:.3f}_{h:.3f}.npz"
    
    if nuc_path.exists():
        data = np.load(str(nuc_path))
        J_brute = float(np.atleast_1d(data["rate_per_site"])[0])
        return J_brute

    raise FileNotFoundError(f"Could not find nucleation rate for beta={beta:.3f}, h={h:.3f}")



In [105]:
h = 0.058
beta = 0.511
L = 64
brute_rate = load_brute_force_rate(beta, h)
print("Brute-force nucleation rate:", brute_rate)

lags = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]

for rc in ["cnn"]:
    print(f"Results for {rc.upper()}:")

    # Build all models first so we can compare tau and 2*tau
    models = {}
    its_dict = {}

    for lagtime in lags:
        C = load_count_matrix(beta, h, rc, lagtime)
        P = count_to_transition(C, normalize_rows=True)
        
        msm = MarkovStateModel(transition_matrix=P, lagtime=lagtime, reversible=False)

        models[lagtime] = msm
        its_dict[lagtime] = np.asarray(msm.timescales()[:3], dtype=float)

    # ITS change to next lag
    its_change_dict = {}
    for i, lagtime in enumerate(lags):
        if i < len(lags) - 1:
            next_lag = lags[i + 1]
            its_now = its_dict[lagtime]
            its_next = its_dict[next_lag]

            n = min(len(its_now), len(its_next))
            rel = np.abs(its_next[:n] - its_now[:n]) / np.maximum(np.abs(its_now[:n]), 1e-300)
            its_change_dict[lagtime] = float(np.mean(rel))
        else:
            its_change_dict[lagtime] = np.nan

    for lagtime in lags:
        msm = models[lagtime]
        P = np.asarray(msm.transition_matrix, dtype=float)

        mfpt = msm.mfpt(0, P.shape[0] - 1)
        rate = 1.0 / mfpt / (L * L)

        its = its_dict[lagtime]
        its_change = its_change_dict[lagtime]

        print(f"\nlagtime {lagtime}")
        print("nucleation rate:", rate)
        print("difference to brute-force rate:", np.abs(rate - brute_rate))
        print("timescales:", its)
        print("ITS change to next lag:", its_change)

        # CK check: compare T(tau)^2 to T(2*tau)
        if 2 * lagtime in models:
            P2_emp = np.asarray(models[2 * lagtime].transition_matrix, dtype=float)
            P2_pred = P @ P
            diff = P2_pred - P2_emp

            ck_max = np.max(np.abs(diff))
            ck_fro = np.linalg.norm(diff)

            print("CK max abs error:", ck_max)
            print("CK Frobenius error:", ck_fro)
        else:
            print("CK check: skipped (2*lagtime not available)")

Brute-force nucleation rate: 1.6122091850036214e-07
Results for CNN:

lagtime 16
nucleation rate: 1.7933862415597315e-07
difference to brute-force rate: 1.811770565561101e-08
timescales: [845.846457 124.263613  43.125267]
ITS change to next lag: 0.12878225393726744
CK max abs error: 0.05027497360995309
CK Frobenius error: 0.09770661949016023

lagtime 32
nucleation rate: 1.6756423143316407e-07
difference to brute-force rate: 6.343312932801928e-09
timescales: [1077.914344  117.549904   45.624682]
ITS change to next lag: 0.05274531324745249
CK max abs error: 0.05622615258009317
CK Frobenius error: 0.0857165840911249

lagtime 64
nucleation rate: 1.636271493464147e-07
difference to brute-force rate: 2.4062308460525487e-09
timescales: [1175.26735   110.451992   45.968589]
ITS change to next lag: 0.05192227709743075
CK max abs error: 0.049248699724757095
CK Frobenius error: 0.09830012016691889

lagtime 128
nucleation rate: 1.5763900523013445e-07
difference to brute-force rate: 3.5819132702276

/home/eng/phunsc/phd/project/commitor/GPU_2DIsing/.venv/lib/python3.11/site-packages/deeptime/markov/tools/analysis/_decomposition.py:496: ImaginaryEigenValueWarning: Using eigenvalues with non-zero imaginary part
  warnings.warn('Using eigenvalues with non-zero imaginary part', ImaginaryEigenValueWarning)
/home/eng/phunsc/phd/project/commitor/GPU_2DIsing/.venv/lib/python3.11/site-packages/deeptime/markov/tools/analysis/_mean_first_passage_time.py:118: RuntimeWarning: invalid value encountered in divide
  muX = nuX / np.sum(nuX)
